# Matcha-TTS Tamil-Only Fine-Tuning (Kaggle)

Train a Tamil-only Matcha model using IndicVoices-R Tamil data, with checkpoint resume and ONNX export.


In [ ]:
# 1) GPU check
!nvidia-smi


In [ ]:
# 2) System deps
!apt-get -qq update
!apt-get -qq install -y espeak-ng ffmpeg sox libsndfile1


In [ ]:
# 3) Python deps + Matcha clone (force coherent NumPy/Pandas ABI)
!pip -q install --upgrade pip setuptools wheel packaging
!pip -q install --no-cache-dir --force-reinstall \
  "numpy>=2.0,<2.3" "pandas>=2.2,<2.4" "pyarrow>=17,<20" \
  "datasets>=4.8,<5" "huggingface_hub>=0.30" "soundfile>=0.12.1" "librosa>=0.10.2"

!python - <<'PY2'
import numpy, pandas, pyarrow, datasets
print('numpy   =', numpy.__version__)
print('pandas  =', pandas.__version__)
print('pyarrow =', pyarrow.__version__)
print('datasets=', datasets.__version__)
PY2

!git clone https://github.com/shivammehta25/Matcha-TTS.git
%cd /kaggle/working/Matcha-TTS


### Important

If you still see `numpy.dtype size changed` after cell 3, restart the Kaggle kernel once and rerun from cell 3 onward.


In [ ]:
# 4) Patch pyproject for Python >= 3.12, then editable install
import sys
from pathlib import Path

p = Path('pyproject.toml')
t = p.read_text(encoding='utf-8')
if sys.version_info >= (3, 12):
    t = t.replace('numpy==1.24.3', 'numpy>=1.26.4')
    t = t.replace('cython==0.29.35', 'cython>=0.29.35')
    p.write_text(t, encoding='utf-8')
    print('Patched pyproject.toml for Python >= 3.12')
else:
    print('No pyproject patch needed')

!pip -q install --no-build-isolation -e . -v


In [ ]:
# 5) Workspace dirs
import os
os.makedirs('/kaggle/working/data_tamil/audio_ta_22k', exist_ok=True)
os.makedirs('/kaggle/working/Matcha-TTS/data/filelists', exist_ok=True)
os.makedirs('/kaggle/working/ckpts/tamil', exist_ok=True)
os.makedirs('/kaggle/working/exports', exist_ok=True)
print('Dirs ready')


In [ ]:
# 6) Hugging Face login + list Tamil parquet files
from huggingface_hub import login, HfApi, hf_hub_download
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
token = user_secrets.get_secret('HF_TOKEN')
assert token, 'HF_TOKEN not found in Kaggle Secrets'

login(token=token)
api = HfApi()
repo_id = 'ai4bharat/indicvoices_r'
files = api.list_repo_files(repo_id=repo_id, repo_type='dataset', token=token)

tamil_files = [f for f in files if f.startswith('Tamil/') and f.endswith('.parquet')]
print('Tamil parquet files:', len(tamil_files))
print(tamil_files[:10])


In [ ]:
# 7) Download Tamil shards (increase N_TAMIL_SHARDS if needed)
N_TAMIL_SHARDS = 7
out_dir = '/kaggle/working/ivr_tamil'
download_list = tamil_files[:N_TAMIL_SHARDS]

for f in download_list:
    p = hf_hub_download(
        repo_id=repo_id,
        repo_type='dataset',
        filename=f,
        token=token,
        local_dir=out_dir,
        local_dir_use_symlinks=False,
    )
    print('Downloaded:', p)

!df -h /kaggle/working


In [ ]:
# 8) Build clean Tamil wav list (22.05k PCM16)
import os, glob, random, re
import numpy as np
import soundfile as sf
from datasets import load_dataset, Audio

random.seed(42)

def to_float_safe(x, default=None):
    if x is None:
        return default
    if isinstance(x, (int, float)):
        return float(x)
    s = str(x).strip()
    m = re.search(r'-?\d+(?:\.\d+)?', s)
    return float(m.group(0)) if m else default

parquets = sorted(glob.glob('/kaggle/working/ivr_tamil/Tamil/train-*.parquet'))
print('parquets:', len(parquets))
assert parquets, 'No parquet files found'

ds = load_dataset('parquet', data_files={'train': parquets}, split='train')
assert 'audio' in ds.column_names, 'audio column missing'
text_col = 'normalized' if 'normalized' in ds.column_names else ('text' if 'text' in ds.column_names else 'verbatim')

# decode/resample with datasets audio feature
ds = ds.cast_column('audio', Audio(sampling_rate=22050))

audio_dir = '/kaggle/working/data_tamil/audio_ta_22k'
TARGET_TAMIL_UTTS = 3500
rows_ta = []
uid = 0
bad = 0

for ex in ds:
    if len(rows_ta) >= TARGET_TAMIL_UTTS:
        break

    txt = str(ex.get(text_col, '')).strip()
    if not txt:
        continue

    d = to_float_safe(ex.get('duration', None), None)
    if d is not None and (d < 1.0 or d > 12.0):
        continue

    snr = to_float_safe(ex.get('snr', None), None)
    if snr is not None and snr < 10:
        continue

    cer = to_float_safe(ex.get('cer', None), None)
    if cer is not None and cer > 0.12:
        continue

    try:
        a = ex['audio']
        y = np.asarray(a['array'], dtype=np.float32)
        if y.ndim > 1:
            y = y.mean(axis=1)
        if y.size == 0:
            bad += 1
            continue

        out_wav = os.path.join(audio_dir, f'ta22k_{uid:07d}.wav')
        uid += 1
        sf.write(out_wav, y, 22050, subtype='PCM_16')

        txt = ' '.join(txt.replace('|', ' ').split())
        rows_ta.append((out_wav, txt))
    except Exception:
        bad += 1

print('Tamil usable:', len(rows_ta), 'bad:', bad)


In [ ]:
# 9) Build Tamil-only filelists
import random
random.seed(42)

random.shuffle(rows_ta)

val_n = min(400, max(100, int(0.1 * len(rows_ta))))
val_rows = rows_ta[:val_n]
train_rows = rows_ta[val_n:]

train_out = '/kaggle/working/Matcha-TTS/data/filelists/ta_train_22k.txt'
val_out = '/kaggle/working/Matcha-TTS/data/filelists/ta_val_22k.txt'

with open(train_out, 'w', encoding='utf-8') as f:
    for ap, tx in train_rows:
        f.write(f'{ap}|{tx}\n')

with open(val_out, 'w', encoding='utf-8') as f:
    for ap, tx in val_rows:
        f.write(f'{ap}|{tx}\n')

print('train:', len(train_rows), train_out)
print('val  :', len(val_rows), val_out)
print('sample lines:')
with open(train_out, 'r', encoding='utf-8') as f:
    for _ in range(3):
        print(f.readline().rstrip())


In [ ]:
# 10) Write Tamil data config
from pathlib import Path

cfg = """_target_: matcha.data.text_mel_datamodule.TextMelDataModule
name: tamil_ivr
train_filelist_path: /kaggle/working/Matcha-TTS/data/filelists/ta_train_22k.txt
valid_filelist_path: /kaggle/working/Matcha-TTS/data/filelists/ta_val_22k.txt
batch_size: 8
num_workers: 2
pin_memory: true
cleaners: [basic_cleaners]
add_blank: true
n_spks: 1
n_fft: 1024
n_feats: 80
sample_rate: 22050
hop_length: 256
win_length: 1024
f_min: 0
f_max: 8000
data_statistics:
  mel_mean: -5.5
  mel_std: 2.0
seed: 42
load_durations: false
"""

p = Path('/kaggle/working/Matcha-TTS/configs/data/tamil.yaml')
p.write_text(cfg, encoding='utf-8')
print('written:', p)


In [ ]:
# 11) Patch symbols.py with chars seen in Tamil filelists
from pathlib import Path

symbols_py = Path('/kaggle/working/Matcha-TTS/matcha/text/symbols.py')
train_txt = Path('/kaggle/working/Matcha-TTS/data/filelists/ta_train_22k.txt')
val_txt = Path('/kaggle/working/Matcha-TTS/data/filelists/ta_val_22k.txt')

def collect_chars(paths):
    out = set()
    for p in paths:
        with p.open('r', encoding='utf-8') as f:
            for line in f:
                if '|' not in line:
                    continue
                _, text = line.rstrip('\n').split('|', 1)
                for ch in text:
                    if ch not in "\n\r\t ":
                        out.add(ch)
    return sorted(out)

chars = collect_chars([train_txt, val_txt])
src = symbols_py.read_text(encoding='utf-8')

force_block = (
    "\n\n# --- TA force symbols (auto-added) ---\n"
    f"_FORCE = {repr(chars)}\n"
    "for _c in _FORCE:\n"
    "    if _c not in symbols:\n"
    "        symbols.append(_c)\n"
)

if '# --- TA force symbols (auto-added) ---' not in src:
    src += force_block
else:
    src = src.split('# --- TA force symbols (auto-added) ---')[0] + force_block

symbols_py.write_text(src, encoding='utf-8')
print('Force-patched symbols.py with', len(chars), 'chars')


In [ ]:
# 12) Verify vocab size for training (add_blank=true -> +1)
from matcha.text import symbols
N_VOCAB = len(symbols) + 1
print('len(symbols)=', len(symbols))
print('N_VOCAB (+blank)=', N_VOCAB)
print('contains அ:', 'அ' in symbols)


In [ ]:
# 13) Compute mel stats and auto-write them to tamil.yaml
%cd /kaggle/working/Matcha-TTS

import subprocess, re, ast
from pathlib import Path

cmd = ['matcha-data-stats', '-i', 'tamil.yaml']
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
print(proc.stderr)

all_txt = (proc.stdout or '') + "\n" + (proc.stderr or '')
m = re.search(r"\{[^\}]*'mel_mean'[^\}]*\}", all_txt)
assert m, 'Could not parse mel stats dict from matcha-data-stats output'
stats = ast.literal_eval(m.group(0))
print('parsed stats:', stats)

p = Path('/kaggle/working/Matcha-TTS/configs/data/tamil.yaml')
t = p.read_text(encoding='utf-8')
t = re.sub(r'mel_mean:\s*[-+]?\d+(?:\.\d+)?', f"mel_mean: {stats['mel_mean']}", t)
t = re.sub(r'mel_std:\s*[-+]?\d+(?:\.\d+)?', f"mel_std: {stats['mel_std']}", t)
p.write_text(t, encoding='utf-8')
print('Updated', p)


In [ ]:
# 14) Patch matplotlib API break in matcha/utils/utils.py
from pathlib import Path

p = Path('/kaggle/working/Matcha-TTS/matcha/utils/utils.py')
s = p.read_text(encoding='utf-8')

old = '''def save_figure_to_numpy(fig):
    # save it to a numpy array
    data = np.fromstring(fig.canvas.tostring_rgb(), dtype=np.uint8, sep="")
    data = data.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    return data
'''
new = '''def save_figure_to_numpy(fig):
    # Matplotlib>=3.8 removed tostring_rgb on FigureCanvasAgg.
    if hasattr(fig.canvas, "tostring_rgb"):
        data = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        data = data.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        return data

    # Fallback path for newer backends: use ARGB and convert to RGB.
    argb = np.frombuffer(fig.canvas.tostring_argb(), dtype=np.uint8)
    argb = argb.reshape(fig.canvas.get_width_height()[::-1] + (4,))
    data = argb[:, :, 1:4]
    return data
'''

if old in s:
    s = s.replace(old, new)
    p.write_text(s, encoding='utf-8')
    print('patched utils.py: save_figure_to_numpy')
else:
    print('utils.py already patched or pattern changed')


In [ ]:
# 15) Patch matplotlib API break in matcha/utils/utils.py
from pathlib import Path

p = Path('/kaggle/working/Matcha-TTS/matcha/utils/utils.py')
s = p.read_text(encoding='utf-8')

old = '''def save_figure_to_numpy(fig):
    data = np.fromstring(fig.canvas.tostring_rgb(), dtype=np.uint8, sep="")
    data = data.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    return data
'''

new = '''def save_figure_to_numpy(fig):
    fig.canvas.draw()
    if hasattr(fig.canvas, "tostring_rgb"):
        data = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        data = data.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        return data
    rgba = np.asarray(fig.canvas.buffer_rgba())
    return np.ascontiguousarray(rgba[..., :3])
'''

if old in s:
    s = s.replace(old, new)
    p.write_text(s, encoding='utf-8')
    print('Patched:', p)
else:
    print('save_figure_to_numpy snippet not found; patch manually if needed')


In [ ]:
# 16) Patch train.py for PyTorch 2.6 checkpoint loading + optional module freezing
from pathlib import Path

p = Path('/kaggle/working/Matcha-TTS/matcha/train.py')
s = p.read_text(encoding='utf-8')

# A) PyTorch 2.6 compatibility for checkpoint restore
old_fit = 'trainer.fit(model=model, datamodule=datamodule, ckpt_path=cfg.get("ckpt_path"))'
new_fit = 'trainer.fit(model=model, datamodule=datamodule, ckpt_path=cfg.get("ckpt_path"), weights_only=False)'
if old_fit in s:
    s = s.replace(old_fit, new_fit)

# B) Optional freeze flags controlled from Hydra CLI:
#    +freeze_encoder=true
#    +freeze_decoder=true
#    +freeze_text_embed=true
marker = '# OPTIONAL_FREEZE_FLAGS_PATCH'
if marker not in s:
    anchor = 'model: LightningModule = hydra.utils.instantiate(cfg.model)\n'
    inject = (
        'model: LightningModule = hydra.utils.instantiate(cfg.model)\n'
        '\n'
        f'{marker}\n'
        'def _set_trainable(module, trainable: bool) -> None:\n'
        '    for param in module.parameters():\n'
        '        param.requires_grad = trainable\n'
        '\n'
        'if cfg.get("freeze_encoder", False):\n'
        '    _set_trainable(model.encoder, False)\n'
        '    log.info("Applied freeze: model.encoder")\n'
        '\n'
        'if cfg.get("freeze_decoder", False):\n'
        '    _set_trainable(model.decoder, False)\n'
        '    log.info("Applied freeze: model.decoder")\n'
        '\n'
        'if cfg.get("freeze_text_embed", False) and hasattr(model.encoder, "emb"):\n'
        '    _set_trainable(model.encoder.emb, False)\n'
        '    log.info("Applied freeze: model.encoder.emb")\n'
    )
    if anchor in s:
        s = s.replace(anchor, inject)
    else:
        raise RuntimeError('Could not find model instantiation anchor in train.py')

p.write_text(s, encoding='utf-8')
print('patched train.py: weights_only=False + freeze flags support')


In [ ]:
# 17) Download English base checkpoint (used as initialization for Tamil fine-tuning)
!mkdir -p /kaggle/working/pretrained
!wget -q -O /kaggle/working/pretrained/matcha_ljspeech.ckpt \
  https://github.com/shivammehta25/Matcha-TTS-checkpoints/releases/download/v1.0/matcha_ljspeech.ckpt
!ls -lh /kaggle/working/pretrained/matcha_ljspeech.ckpt


In [ ]:
# 18) Stage-1 fine-tuning (short warmup with decoder frozen)
%cd /kaggle/working/Matcha-TTS

!python matcha/train.py \
  experiment=ljspeech \
  data=tamil \
  model.n_vocab={N_VOCAB} \
  ckpt_path=/kaggle/working/pretrained/matcha_ljspeech.ckpt \
  +freeze_decoder=true \
  trainer.devices=1 \
  trainer.accelerator=gpu \
  trainer.precision=16-mixed \
  trainer.max_epochs=8 \
  trainer.check_val_every_n_epoch=1 \
  extras.print_config=false \
  test=false \
  ~callbacks.model_summary \
  ~callbacks.rich_progress_bar \
  +trainer.log_every_n_steps=100 \
  +trainer.accumulate_grad_batches=2 \
  callbacks.model_checkpoint.dirpath=/kaggle/working/ckpts/tamil \
  "callbacks.model_checkpoint.filename='ta_epoch_{epoch:02d}'" \
  callbacks.model_checkpoint.save_last=true \
  callbacks.model_checkpoint.save_top_k=5 \
  "callbacks.model_checkpoint.monitor='loss/val_epoch'" \
  callbacks.model_checkpoint.mode=min \
  callbacks.model_checkpoint.every_n_epochs=1 \
  callbacks.model_checkpoint.verbose=false \
  data.batch_size=8


In [ ]:
# 19) Stage-2 fine-tuning (unfreeze all + robust resume from last.ckpt)
%cd /kaggle/working/Matcha-TTS

!CKPT=$(test -f /kaggle/working/ckpts/tamil/last.ckpt && echo /kaggle/working/ckpts/tamil/last.ckpt || echo /kaggle/working/pretrained/matcha_ljspeech.ckpt); \
echo "Using ckpt: $CKPT"; \
python matcha/train.py \
  experiment=ljspeech \
  data=tamil \
  model.n_vocab={N_VOCAB} \
  ckpt_path="$CKPT" \
  +freeze_decoder=false \
  +freeze_encoder=false \
  +freeze_text_embed=false \
  trainer.devices=1 \
  trainer.accelerator=gpu \
  trainer.precision=16-mixed \
  trainer.max_epochs=60 \
  trainer.check_val_every_n_epoch=1 \
  extras.print_config=false \
  test=false \
  ~callbacks.model_summary \
  ~callbacks.rich_progress_bar \
  +trainer.log_every_n_steps=100 \
  +trainer.accumulate_grad_batches=2 \
  callbacks.model_checkpoint.dirpath=/kaggle/working/ckpts/tamil \
  "callbacks.model_checkpoint.filename='ta_epoch_{epoch:02d}'" \
  callbacks.model_checkpoint.save_last=true \
  callbacks.model_checkpoint.save_top_k=5 \
  "callbacks.model_checkpoint.monitor='loss/val_epoch'" \
  callbacks.model_checkpoint.mode=min \
  callbacks.model_checkpoint.every_n_epochs=1 \
  callbacks.model_checkpoint.verbose=false \
  data.batch_size=8


In [ ]:
# 20) Show best checkpoint from ModelCheckpoint metadata
import torch

ckpt_path = '/kaggle/working/ckpts/tamil/last.ckpt'
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)

mc = None
for k, v in ckpt.get('callbacks', {}).items():
    if 'ModelCheckpoint' in k:
        mc = v
        break

assert mc is not None, 'ModelCheckpoint state not found'
print('best_model_path :', mc.get('best_model_path'))
print('best_model_score:', float(mc.get('best_model_score')))
print('last_model_path :', mc.get('last_model_path'))
print('\nTop-K (lower is better):')
for pth, sc in sorted(mc.get('best_k_models', {}).items(), key=lambda x: float(x[1])):
    print(f"{float(sc):.6f}  {pth}")


In [ ]:
# 21) Patch cli.py for PyTorch 2.6 checkpoint loading (ONNX export path)
from pathlib import Path

p = Path('/kaggle/working/Matcha-TTS/matcha/cli.py')
s = p.read_text(encoding='utf-8')
old = 'model = MatchaTTS.load_from_checkpoint(checkpoint_path, map_location=device)'
new = 'model = MatchaTTS.load_from_checkpoint(checkpoint_path, map_location=device, weights_only=False)'

if old in s:
    s = s.replace(old, new)
    p.write_text(s, encoding='utf-8')
    print('patched cli.py')
else:
    print('cli.py already patched or pattern changed')


In [ ]:
# 22) Patch ONNX export for torch>=2.6 (force legacy exporter path)
from pathlib import Path

p = Path('/kaggle/working/Matcha-TTS/matcha/onnx/export.py')
s = p.read_text(encoding='utf-8')

if 'dynamo=False' not in s:
    s = s.replace(
        'do_constant_folding=True,\n    )',
        'do_constant_folding=True,\n        dynamo=False,\n    )'
    )
    p.write_text(s, encoding='utf-8')
    print('patched: dynamo=False added')
else:
    print('already patched')

!grep -n "dynamo=False\|to_onnx" /kaggle/working/Matcha-TTS/matcha/onnx/export.py


In [ ]:
# 23) Export best Tamil checkpoint to ONNX (3 steps)
BEST_CKPT = '/kaggle/working/ckpts/tamil/last.ckpt'
OUT = '/kaggle/working/exports/matcha_tamil_t3.onnx'

%cd /kaggle/working/Matcha-TTS
!python -m matcha.onnx.export \
  {BEST_CKPT} \
  {OUT} \
  --n-timesteps 3

!ls -lh {OUT}


In [ ]:
# 24) Optional E2E ONNX (embedded vocoder)
%cd /kaggle/working/Matcha-TTS
!mkdir -p /kaggle/working/vocoder
!wget -q -O /kaggle/working/vocoder/g_02500000 \
  https://github.com/shivammehta25/Matcha-TTS-checkpoints/releases/download/v1.0/g_02500000

BEST_CKPT = '/kaggle/working/ckpts/tamil/last.ckpt'
OUT_E2E = '/kaggle/working/exports/matcha_tamil_e2e_t3.onnx'

!python -m matcha.onnx.export \
  {BEST_CKPT} \
  {OUT_E2E} \
  --n-timesteps 3 \
  --vocoder-name hifigan_univ_v1 \
  --vocoder-checkpoint-path /kaggle/working/vocoder/g_02500000

!ls -lh {OUT_E2E}


In [ ]:
# 25) Quick Tamil inference test (E2E ONNX)
%cd /kaggle/working/Matcha-TTS
!python -m matcha.onnx.infer \
  /kaggle/working/exports/matcha_tamil_e2e_t3.onnx \
  --text "வணக்கம் இது தமிழ் குரல் சோதனை." \
  --output-dir /kaggle/working/exports/infer_ta

!ls -lah /kaggle/working/exports/infer_ta


In [ ]:
# 26) Bundle artifacts before runtime ends
!zip -r /kaggle/working/matcha_tamil_artifacts.zip \
  /kaggle/working/ckpts/tamil \
  /kaggle/working/exports \
  /kaggle/working/Matcha-TTS/configs/data/tamil.yaml \
  /kaggle/working/Matcha-TTS/data/filelists/ta_train_22k.txt \
  /kaggle/working/Matcha-TTS/data/filelists/ta_val_22k.txt

!ls -lh /kaggle/working/matcha_tamil_artifacts.zip
